Project Name - Gender Classification Model

Project Type - Productionization of ML Systems

Contribution - Individual

Name of Contributor - Meetu Kumari

GitHub Link -  https://github.com/MeetuKumari1/ml_productionization.git

Brief overview of dataset

Hotels Dataset:

* travelCode: Identifier for the travel, similar to the Flights dataset.
* userCode: User identifier(linked to the Users dataset)
* name: Name of the hotel.
* place: Location of the hotel.
* days: Number of days of the hotel stay.
* price: Price per day.
* total: Total price for the stay.
* date: Date of the hotel booking.

Project Objectives

Build a recommendation model to provide hotel suggestions based on user preferences and historical data.

In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
HOTELS_PATH = Path("travel_capstone/hotels.csv")
USERS_PATH = Path("travel_capstone/users.csv")
ARTIFACT_PATH = Path("artifacts/hotel_recommender.joblib")
METADATA_PATH = Path("artifacts/hotel_recommender_metadata.json")

In [4]:
def load_hotels(data_path: str | Path = HOTELS_PATH) -> pd.DataFrame:
    path = Path(data_path)
    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}")
    return pd.read_csv(path)

In [5]:
def build_recommender(df: pd.DataFrame) -> dict:
    if df[["userCode", "name"]].isna().any().any():
        raise ValueError("Missing values in userCode/name.")

    interactions = pd.crosstab(df["userCode"], df["name"])
    user_ids = interactions.index.to_list()
    hotel_names = interactions.columns.to_list()

    item_matrix = interactions.T.values
    item_similarity = cosine_similarity(item_matrix)

    hotel_stats = (
        df.groupby(["name", "place"], as_index=False)
        .agg(bookings=("travelCode", "count"), avg_price=("price", "mean"))
        .sort_values(["bookings", "avg_price"], ascending=[False, True])
    )

    return {
        "user_ids": user_ids,
        "hotel_names": hotel_names,
        "item_similarity": item_similarity,
        "interactions": interactions,
        "hotel_stats": hotel_stats,
    }

In [6]:
def save_recommender(recommender: dict) -> dict:
    ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(recommender, ARTIFACT_PATH)

    metadata = {
        "artifact_path": str(ARTIFACT_PATH),
        "num_users": len(recommender["user_ids"]),
        "num_hotels": len(recommender["hotel_names"]),
    }
    METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    return metadata


In [7]:
def train_recommender(data_path: str | Path = HOTELS_PATH) -> dict:
    df = load_hotels(data_path)
    recommender = build_recommender(df)
    return save_recommender(recommender)

In [8]:
def load_recommender() -> dict:
    if not ARTIFACT_PATH.exists():
        raise FileNotFoundError(
            f"Recommender artifact not found at {ARTIFACT_PATH}. "
            "Run train_recommender() first."
        )
    return joblib.load(ARTIFACT_PATH)


In [9]:
def recommend_for_user(recommender: dict, user_code: int,top_n: int = 5,) -> pd.DataFrame:
    interactions = recommender["interactions"]
    hotel_stats = recommender["hotel_stats"]
    hotel_names = recommender["hotel_names"]
    item_similarity = recommender["item_similarity"]

    if user_code not in interactions.index:
        return hotel_stats.head(top_n)

    user_vector = interactions.loc[user_code].values
    scores = user_vector @ item_similarity
    seen_mask = user_vector > 0
    scores = scores.astype(float)
    scores[seen_mask] = -np.inf

    top_idx = np.argsort(scores)[::-1][:top_n]
    recommended_hotels = [hotel_names[i] for i in top_idx]

    return hotel_stats[hotel_stats["name"].isin(recommended_hotels)]

In [10]:
def main():
    metadata = train_recommender()
    print("Recommender trained and saved.")
    print(metadata)


if __name__ == "__main__":
    main()

Recommender trained and saved.
{'artifact_path': 'artifacts\\hotel_recommender.joblib', 'num_users': 1310, 'num_hotels': 9}
